# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset Croissant schema is available at:
[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Display main metadata
metadata = dataset.metadata  # This is NOT a dict; don't subscript!
print('Dataset Title:', getattr(metadata, 'name', ''))
print('Description:', getattr(metadata, 'description', ''))

# Print publish date and version
print('Published:', getattr(metadata, 'datePublished', ''))
print('Version:', getattr(metadata, 'version', ''))

## 2. Data Overview
Discover available record sets and their fields, columns, and Croissant `@id`s.

- <b>Note</b>: All references to record sets, fields or columns should use their Croissant `@id` for full reproducibility and exact referencing.

In [ ]:
# List available record sets and fields in schema (by @id)

record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []
if isinstance(record_sets, dict):
    record_sets = [record_sets]
if not record_sets:
    print('No record sets found directly in metadata; attempting to extract via schema.')
    # Try extracting with .to_json().get('recordSet', ...):
    js = dataset.metadata.to_json()
    record_sets = js.get('recordSet', [])
    if isinstance(record_sets, dict):
        record_sets = [record_sets]
# Display record set @id and their fields
overview = {}
for rset in record_sets:
    rset_id = rset.get('@id', None) if isinstance(rset, dict) else getattr(rset, '@id', None)
    print(f'RecordSet @id: {rset_id}')
    # Fields
    fields = rset.get('field', []) if isinstance(rset, dict) else getattr(rset, 'field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        field_id = field.get('@id', None) if isinstance(field, dict) else getattr(field, '@id', None)
        print(f'  Field @id: {field_id}')
        if hasattr(field, 'column') or ('column' in field if isinstance(field, dict) else False):
            columns = getattr(field, 'column', []) if hasattr(field, 'column') else field.get('column', [])
            if isinstance(columns, dict):
                columns = [columns]
            for col in columns:
                col_id = col.get('@id', None) if isinstance(col, dict) else getattr(col, '@id', None)
                print(f'    Column @id: {col_id}')

### Example record: print first few from each available record set


In [ ]:
# List all record set @ids for extraction
record_set_ids = []
# The exact @id can be gotten by examining the Croissant schema
try:
    # prefer using to_json for consistent parsing
    js = dataset.metadata.to_json()
    record_sets = js.get('recordSet', [])
    if isinstance(record_sets, dict):
        record_sets = [record_sets]
    for rset in record_sets:
        rid = rset.get('@id')
        if rid:
            record_set_ids.append(rid)
    if not record_set_ids:
        print('No record set @id found.')
    else:
        print('RecordSet @ids:', record_set_ids)
except Exception as e:
    print('Could not parse recordSets:', e)

# Show first records for each available record set
for rid in record_set_ids:
    print(f'First 2 rows for RecordSet @id={rid}:')
    try:
        for i, rec in enumerate(dataset.records(record_set=rid)):
            if i >= 2:
                break
            print(rec)
    except Exception as e:
        print(f'Error loading for {rid}: {e}')
    print('---')

## 3. Data Extraction
Load data for each record set into a pandas DataFrame for further analysis.
Refer to their `@id` for identifiability.

In [ ]:
dataframes = {}
# Download each record set and convert to DataFrame
for rid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rid))
        if len(records) > 0:
            dataframes[rid] = pd.DataFrame(records)
            print(f'Loaded RecordSet {rid} with {len(records)} records.')
        else:
            print(f'No records found for {rid}.')
    except Exception as e:
        print(f'Error loading records for {rid}: {e}')

# Pick the main data record set for demo. Common @id in survey datasets is 'survey_responses' or similar.
main_record_set_id = None
for rid in dataframes.keys():
    # Heuristically: main table has most variables/columns
    if main_record_set_id is None:
        main_record_set_id = rid
    if 'survey' in rid.lower() or 'kilifi' in rid.lower():
        main_record_set_id = rid

if main_record_set_id is not None:
    print(f'Using main data RecordSet @id: {main_record_set_id}')
    print('Columns:', dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No main data RecordSet could be identified.')

## 4. Exploratory Data Analysis (EDA)
We'll process and analyze columns using their `@id` identifiers.

Typical EDA steps:
 - Remove outliers
 - Filter records
 - Normalize numerical fields
 - Group by categorical fields

<b>Note</b>: Replace the field `@id`s below as appropriate based on schema exploration above.

In [ ]:
# Choose a numeric and category field by their exact @id (edit as needed!)
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Example: GAD-7 total score (edit field/column @id as needed as per EDA above)
    possible_numeric_fields = [c for c in df.columns if 'score' in c.lower() or 'gad' in c.lower() or 'phq' in c.lower() or 'psq' in c.lower()]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]  # e.g., '@id': 'gad7_score'
    else:
        numeric_field = df.columns[0]
    print('Numeric field chosen (@id):', numeric_field)
    
    # Example group: e.g., Gender or level of education, both likely present and marked by their own @id
    possible_group_fields = [c for c in df.columns if 'gender' in c.lower() or 'education' in c.lower()]
    group_field = possible_group_fields[0] if possible_group_fields else None

    # Drop missing
    filtered_df = df[df[numeric_field].notnull()].copy()

    # Use a threshold for demo; e.g., GAD7 > 7
    threshold = 7
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f'Filtered records where {numeric_field} > {threshold}:')
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field}:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field is not None and group_field in filtered_df.columns:
        print(f'Grouping by: {group_field}')
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f'Mean {numeric_field} by {group_field}:')
        display(grouped_df)
else:
    print('No main data record set available for EDA.')

## 5. Visualization
Visualize distribution of the chosen numeric field and by category group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    possible_numeric_fields = [c for c in df.columns if 'score' in c.lower() or 'gad' in c.lower() or 'phq' in c.lower() or 'psq' in c.lower()]
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]
    possible_group_fields = [c for c in df.columns if 'gender' in c.lower() or 'education' in c.lower()]
    group_field = possible_group_fields[0] if possible_group_fields else None
    
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True, color='dodgerblue')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f'{numeric_field} by {group_field}')
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion
We have loaded, explored, and conducted initial processing of the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

- **Metadata and schema** were programmatically inspected by Croissant `@id` for clear lineages.
- **Data extraction** was performed for all record sets.
- **Basic EDA** covered filtering and normalization of numeric mental health indicator fields, plus grouping by demographics.
- **Visualization** allows understanding score distributions and demographic patterns.

For advanced ML-ready workflows, continue by cross-referencing full Croissant field `@id` mappings in your code for reproducible pipelines.

> Dataset license: [Open Data Commons Attribution](https://opendatacommons.org/licenses/by/1-0/)
